In [7]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# Load data
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')
popular_movies = pd.read_csv('../data/popular_movies.csv')

# Load saved models
with open('../data/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

with open('../data/cosine_sim.pkl', 'rb') as f:
    cosine_sim = pickle.load(f)

# Load movie indices
indices = pd.read_csv('../data/movie_indices.csv', index_col=0).squeeze()

print("All models loaded successfully!")

All models loaded successfully!


In [8]:
#Rebuild content based function
def get_content_scores(title, top_n=20):
    """
    Returns similarity scores for movies similar to given title.
    Returns more than top_n so hybrid has enough to work with.
    """
    if title not in indices:
        return None
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    similarity_scores = [i[1] for i in sim_scores]
    
    result = movies.iloc[movie_indices][['movieId', 'title', 'genres']].copy()
    result['content_score'] = similarity_scores
    
    return result

In [9]:
#Hybrid Recommendation function
def hybrid_recommendations(user_id=None, title=None, top_n=10):
    """
    Hybrid recommender that combines collaborative + content based filtering.
    
    Cases:
    1. user_id + title → Full hybrid (collaborative + content)
    2. title only → Content based only (new user / cold start)
    3. user_id only → Collaborative only
    4. neither → Popularity based
    """
    
    # Case 4: No input at all → return popular movies
    if user_id is None and title is None:
        return popular_movies[['title', 'genres', 'weighted_score']].head(top_n)
    
    # Case 2: Only title → Content based (cold start)
    if user_id is None and title is not None:
        content_recs = get_content_scores(title, top_n=top_n)
        if content_recs is None:
            return f"Movie '{title}' not found."
        content_recs = content_recs.merge(
            popular_movies[['movieId', 'weighted_score']],
            on='movieId', how='left'
        )
        content_recs['weighted_score'] = content_recs['weighted_score'].fillna(0)
        content_recs = content_recs.sort_values(
            ['content_score', 'weighted_score'], ascending=[False, False]
        )
        return content_recs[['title', 'genres', 'content_score']].head(top_n)
    
    # Case 3: Only user_id → Collaborative only
    if user_id is not None and title is None:
        rated_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()
        all_movie_ids = movies['movieId'].tolist()
        unseen_movies = [m for m in all_movie_ids if m not in rated_movies]
        
        preds = []
        for movie_id in unseen_movies:
            pred = svd_model.predict(user_id, movie_id)
            preds.append({'movieId': movie_id, 'collab_score': round(pred.est, 3)})
        
        pred_df = pd.DataFrame(preds).sort_values('collab_score', ascending=False)
        result = pred_df.merge(movies, on='movieId')
        return result[['title', 'genres', 'collab_score']].head(top_n)
    
    # Case 1: Both user_id + title → TRUE HYBRID
    if user_id is not None and title is not None:
        
        # Step 1: Get content based scores
        content_recs = get_content_scores(title, top_n=50)
        if content_recs is None:
            return f"Movie '{title}' not found."
        
        # Step 2: Get collaborative scores for those movies
        collab_scores = []
        for _, row in content_recs.iterrows():
            pred = svd_model.predict(user_id, row['movieId'])
            collab_scores.append(round(pred.est, 3))
        
        content_recs['collab_score'] = collab_scores
        
        # Step 3: Normalize both scores to 0-1 range
        content_recs['content_norm'] = (
            content_recs['content_score'] - content_recs['content_score'].min()
        ) / (content_recs['content_score'].max() - content_recs['content_score'].min() + 1e-9)
        
        content_recs['collab_norm'] = (
            content_recs['collab_score'] - content_recs['collab_score'].min()
        ) / (content_recs['collab_score'].max() - content_recs['collab_score'].min() + 1e-9)
        
        # Step 4: Weighted combination
        # 40% content + 60% collaborative
        # Collaborative gets more weight because it's personalized
        content_recs['hybrid_score'] = (
            0.4 * content_recs['content_norm'] +
            0.6 * content_recs['collab_norm']
        )
        
        # Step 5: Sort by hybrid score
        content_recs = content_recs.sort_values('hybrid_score', ascending=False)
        
        return content_recs[['title', 'genres', 'content_score', 
                              'collab_score', 'hybrid_score']].head(top_n)

In [10]:
#Test All 4 cases
# Case 1: Full Hybrid — user + movie
print("=" * 60)
print("CASE 1: Full Hybrid (User 1 + Toy Story)")
print("=" * 60)
print(hybrid_recommendations(user_id=1, title='Toy Story (1995)'))

# Case 2: Cold Start — only movie
print("\n" + "=" * 60)
print("CASE 2: Cold Start (No user, only Toy Story)")
print("=" * 60)
print(hybrid_recommendations(title='Toy Story (1995)'))

# Case 3: Collaborative only — only user
print("\n" + "=" * 60)
print("CASE 3: Collaborative Only (User 1, no movie)")
print("=" * 60)
print(hybrid_recommendations(user_id=1))

# Case 4: Popularity fallback — no input
print("\n" + "=" * 60)
print("CASE 4: Popularity Fallback (No input)")
print("=" * 60)
print(hybrid_recommendations())

CASE 1: Full Hybrid (User 1 + Toy Story)
                                                  title  \
2355                                 Toy Story 2 (1999)   
3568                              Monsters, Inc. (2001)   
8219                                       Turbo (2013)   
5546  Kiki's Delivery Service (Majo no takkyûbin) (1...   
3000                   Emperor's New Groove, The (2000)   
9430                                       Moana (2016)   
8900                                  Inside Out (2015)   
9369                    Kubo and the Two Strings (2016)   
3194                                       Shrek (2001)   
8927                           The Good Dinosaur (2015)   

                                                 genres  content_score  \
2355        Adventure|Animation|Children|Comedy|Fantasy       1.000000   
3568        Adventure|Animation|Children|Comedy|Fantasy       1.000000   
8219        Adventure|Animation|Children|Comedy|Fantasy       1.000000   
5546         

In [11]:
#Evaluation: Precision@K

def precision_at_k(user_id, k=10, threshold=4.0):
    """
    Precision@K using leave-one-out approach:
    1. Take movies user HAS rated
    2. Split into 80% known, 20% hidden
    3. Predict ratings for hidden movies
    4. Check how many of top K predictions are actually good (>=threshold)
    """
    user_ratings = ratings[ratings['userId'] == user_id].copy()
    
    # Need at least 10 rated movies to evaluate properly
    if len(user_ratings) < 10:
        return None
    
    # Split user's ratings: 80% train, 20% test
    test_ratings = user_ratings.sample(frac=0.2, random_state=42)
    
    # Relevant = test movies user actually rated >= threshold
    relevant_movies = set(
        test_ratings[test_ratings['rating'] >= threshold]['movieId'].tolist()
    )
    
    if len(relevant_movies) == 0:
        return None
    
    # Predict ratings for test movies
    preds = []
    for _, row in test_ratings.iterrows():
        pred = svd_model.predict(user_id, row['movieId'])
        preds.append((row['movieId'], pred.est))
    
    # Sort by predicted rating, take top K
    preds.sort(key=lambda x: x[1], reverse=True)
    top_k_movies = set([p[0] for p in preds[:k]])
    
    # How many of our top K predictions are actually relevant?
    hits = len(top_k_movies & relevant_movies)
    precision = hits / k
    
    return precision

# Evaluate on more users
print("Precision@10 for sample users:")
print("-" * 30)

precisions = []
for uid in range(1, 51):  # Test on first 50 users
    p = precision_at_k(uid, k=10, threshold=4.0)
    if p is not None:
        precisions.append(p)

print(f"Users evaluated: {len(precisions)}")
print(f"Average Precision@10: {np.mean(precisions):.3f}")
print(f"Max Precision@10: {np.max(precisions):.3f}")
print(f"Min Precision@10: {np.min(precisions):.3f}")
print(f"\nInterpretation: On average {np.mean(precisions)*100:.1f}% of our")
print(f"top 10 recommendations are movies the user would rate >= 4.0")

Precision@10 for sample users:
------------------------------
Users evaluated: 50
Average Precision@10: 0.660
Max Precision@10: 1.000
Min Precision@10: 0.100

Interpretation: On average 66.0% of our
top 10 recommendations are movies the user would rate >= 4.0


In [12]:
# Save popular movies (already saved but let's make sure)
popular_movies.to_csv('../data/popular_movies.csv', index=False)
print("All files ready for Streamlit app!")
print("\nFiles in data folder:")
import os
for f in os.listdir('../data'):
    print(f" - {f}")

All files ready for Streamlit app!

Files in data folder:
 - cosine_sim.pkl
 - genre_distribution.png
 - links.csv
 - movies.csv
 - movie_indices.csv
 - popular_movies.csv
 - ratings.csv
 - rating_distribution.png
 - README.txt
 - svd_model.pkl
 - tags.csv
